# 用 Transformers 跑 gpt-oss

> 原文：[articles/gpt-oss/run-transformers.md](https://github.com/openai/openai-cookbook/blob/main/articles/gpt-oss/run-transformers.md)



## 這篇在做什麼
示範用 Hugging Face Transformers 在本機或伺服器上跑 gpt-oss-20b 和 gpt-oss-120b。從最簡單的 `pipeline`，到直接呼叫 `.generate()`，再到用 `transformers serve` 開一個相容 Responses API 的服務，還有用 Harmony 格式組 prompt 和多 GPU 跑法。

## 學完你會
- 依 VRAM 挑對模型，並裝好 MXFP4 需要的套件。
- 用 `pipeline` 和 `.generate()` 兩種層級跑 gpt-oss，並用 chat template 或 `openai-harmony` 組 prompt。
- 用 `transformers serve` 開出 Responses / Chat Completions endpoint，並用 `tp_plan="auto"` 跑多 GPU。

## 開跑前
- 需要一張夠大的 GPU（20b 約 16GB VRAM，120b 要 60GB 以上）。MXFP4 要 Hopper 以後的架構。
- 在 Colab 上跑的話，在 cell 前面加 `!` 執行 pip 指令，裝完重啟 runtime，並選有 GPU 的 runtime。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U openai_harmony torch transformers
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set articles/gpt-oss
os.chdir('openai-cookbook/articles/gpt-oss')

# 用 Hugging Face Transformers 跑 gpt-oss

Hugging Face 的 Transformers library 提供彈性的方式，在本機或伺服器上載入並執行大型語言模型。這份指南會帶你用 Transformers 跑 [OpenAI gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b) 或 [OpenAI gpt-oss-120b](https://huggingface.co/openai/gpt-oss-120b)，可以用高階的 pipeline，也可以用低階的 `generate` 直接丟 token ID。

內容包含用高階 pipeline 抽象跑模型、低階 \`generate\` 呼叫，以及用 \`transformers serve\` 在本機架設相容 Responses API 的服務。

這份指南會介紹幾種優化過的方式，透過 Transformers 跑 **gpt-oss 模型。**

加碼：Transformers 也能拿來 fine-tune 模型，[fine-tuning 指南在這裡](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transformers)。

## 挑模型

Hugging Face 上有兩個 **gpt-oss** 模型：

- **`openai/gpt-oss-20b`**
  - 用 MXFP4 約需 \~16GB VRAM
  - 很適合單張高階消費級 GPU
- **`openai/gpt-oss-120b`**
  - 需要 ≥60GB VRAM 或多 GPU
  - 適合 H100 等級的硬體

兩個模型預設都是 **MXFP4 量化**。注意 MXFP4 要 Hopper 以後的架構才支援，包含 H100、GB200 這類資料中心 GPU，以及最新的 RTX 50xx 系列消費級顯卡。

如果改用 `bfloat16` 而不是 MXFP4，記憶體用量會變大（20b 模型約 \~48 GB）。

## 快速設定

1. **安裝相依套件**  
   建議開一個全新的 Python 環境。安裝 transformers、accelerate，還有 MXFP4 相容需要的 Triton kernels（Colab 上在指令前面加 `!` 就能跑）：

```bash
pip install -U transformers accelerate torch triton==3.4 kernels
```

2. **（選用）啟用多 GPU**  
   跑大模型的話，用 Accelerate 或 torchrun 自動處理 device mapping。

## 建立相容 Open AI Responses / Chat Completions 的 endpoint

要啟動 server，用 `transformers serve` 這個 CLI 指令就好：

```bash
transformers serve
```

最簡單的互動方式是用 transformers chat CLI

```bash
transformers chat localhost:8000 --model-name-or-path openai/gpt-oss-20b
```

或是用 cURL 送 HTTP request，例如

```bash
curl -X POST http://localhost:8000/v1/responses -H "Content-Type: application/json" -d '{"messages": [{"role": "system", "content": "hello"}], "temperature": 0.9, "max_tokens": 1000, "stream": true, "model": "openai/gpt-oss-20b"}'
```

其他用法，像是把 `transformers serve` 接上 Cursor 等工具，詳見[文件](https://huggingface.co/docs/transformers/main/serving)。

## 用 pipeline 快速推論

跑 gpt-oss 最簡單的方式是用 Transformers 的高階 `pipeline` API：

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="openai/gpt-oss-20b",
    torch_dtype="auto",
    device_map="auto"  # Automatically place on available GPUs
)

messages = [
    {"role": "user", "content": "Explain what MXFP4 quantization is."},
]

result = generator(
    messages,
    max_new_tokens=200,
    temperature=1.0,
)

print(result[0]["generated_text"])

## 用 `.generate()` 做進階推論

想要更多控制的話，可以手動載入模型和 tokenizer，再呼叫 `.generate()` 方法：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {"role": "user", "content": "Explain what MXFP4 quantization is."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7
)

print(tokenizer.decode(outputs[0]))

## Chat template 和 tool calling

OpenAI gpt-oss 模型用 [harmony response format](https://cookbook.openai.com/article/harmony) 來組織訊息，包含 reasoning 和 tool call。

組 prompt 可以用 Transformers 內建的 chat template。想要更多控制的話，也可以安裝 [openai-harmony library](https://github.com/openai/harmony) 來用。

用 chat template 的方式：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
)

messages = [
    {"role": "system", "content": "Always respond in riddles"},
    {"role": "user", "content": "What is the weather like in Madrid?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1] :]))

要用 [`openai-harmony`](https://github.com/openai/harmony) library 來準備 prompt 和解析回應，先這樣安裝：

```bash
pip install openai-harmony
```

下面是用這個 library 組 prompt 並編碼成 token 的範例：

In [ ]:
import json
from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    Conversation,
    Message,
    Role,
    SystemContent,
    DeveloperContent
)
from transformers import AutoModelForCausalLM, AutoTokenizer

encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

# Build conversation
convo = Conversation.from_messages([
    Message.from_role_and_content(Role.SYSTEM, SystemContent.new()),
    Message.from_role_and_content(
        Role.DEVELOPER,
        DeveloperContent.new().with_instructions("Always respond in riddles")
    ),
    Message.from_role_and_content(Role.USER, "What is the weather like in SF?")
])

# Render prompt
prefill_ids = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)
stop_token_ids = encoding.stop_tokens_for_assistant_actions()

# Load model
model_name = "openai/gpt-oss-20b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

# Generate
outputs = model.generate(
    input_ids=[prefill_ids],
    max_new_tokens=128,
    eos_token_id=stop_token_ids
)

# Parse completion tokens
completion_ids = outputs[0][len(prefill_ids):]
entries = encoding.parse_messages_from_completion_tokens(completion_ids, Role.ASSISTANT)

for message in entries:
    print(json.dumps(message.to_dict(), indent=2))

注意 Harmony 裡的 `Developer` role 對應到 chat template 的 `system` prompt。

## 多 GPU 與分散式推論

大的 gpt-oss-120b 用 MXFP4 的話，單張 H100 就放得下。想用多張 GPU 跑的話，可以：

- 用 `tp_plan="auto"` 自動配置並做 tensor parallelism
- 用 `accelerate launch or torchrun` 啟動分散式
- 利用 Expert Parallelism
- 用特製的 Flash attention kernels 加速推論

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.distributed import DistributedConfig
import torch

model_path = "openai/gpt-oss-120b"
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")

device_map = {
    # Enable Expert Parallelism
    "distributed_config": DistributedConfig(enable_expert_parallel=1),
    # Enable Tensor Parallelism
    "tp_plan": "auto",
}

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    attn_implementation="kernels-community/vllm-flash-attn3",
    **device_map,
)

messages = [
     {"role": "user", "content": "Explain how expert parallelism works in large language models."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1000)

# Decode and print
response = tokenizer.decode(outputs[0])
print("Model response:", response.split("<|channel|>final<|message|>")[-1].strip())

然後在有四張 GPU 的節點上這樣跑：

```bash
torchrun --nproc_per_node=4 generate.py
```

## 重點整理
- Transformers 提供三種層級：`pipeline` 最簡單，`.generate()` 控制最多，`transformers serve` 則直接變成 API 服務。
- gpt-oss 預設是 MXFP4 量化，需要 Hopper 以後的 GPU，用 `bfloat16` 記憶體會大很多。
- Prompt 必須符合 Harmony 格式，可以靠 chat template 或 `openai-harmony`，Harmony 的 `Developer` 就是 `system`。
- 大模型可以用 `tp_plan="auto"` 搭配 `torchrun` 分散到多張 GPU。

## 練習
1. 把 `pipeline` 範例裡的模型從 20b 換成 120b（如果 VRAM 夠），比較輸出和載入時間。
2. 用 chat template 範例，在 messages 裡加一則 `system` 指令（例如「只用英文回答」），看輸出有什麼改變。
3. 用 `transformers serve` 開 server，再用 OpenAI Python SDK 把 `base_url` 指到 `http://localhost:8000/v1` 去呼叫它。